In [1]:
# 図4.39の実装を行う。
# 図4.39は図4.38と同じ。
distances = [[5, [4, 5]], [7, [2, 3]], [6, [1, 3]], [8, [2, 4]], [4, [3, 4]], [3, [3, 5]], [2, [1, 2]]]
nodeNum = 5

# Sは空ではなく、最初に4番目のノード(インデックスは0から始まるので、3)を入れておく。
firstNode = 3


In [2]:
import numpy as np
import copy

# プリム法を実装する。

# step1各種初期化
S = [0 if i != firstNode else 1 for i in range(nodeNum)]
numS = 1
fvs = [[np.inf, []] for i in range(nodeNum)]

resultT = []

copiedDistances = copy.deepcopy(distances)

# fvを更新して、各ノードごとにδ(S)の最小を求めておく。
def updateFvs(addedIndex):
    # まずはすべてのエッジを走査する
    # オーダが|E|これは|V|^2相当になる。
    # addedIndexが新たにSに追加されたノード。これに関連するところだけを更新すればよい。
    deleteList = []

    for i, d in enumerate(copiedDistances):
        try:
            index = d[1].index(addedIndex + 1)
            fvsIndex = (d[1][0] - 1) if index == 1 else (d[1][1] - 1)
            #print(fvsIndex)
            
            # 最小全域木が一意になるとは限らない。(例えば全結合ですべてのエッジの距離が等しい場合)
            # Sをフラグ形式にしないと時間量が1のオーダから、|V|のオーダに増える。
            # addedIndexのノードが追加されたことでそこにつながるノードの距離をチェックし、その値が小さければ更新する。
            if fvs[fvsIndex][0] > d[0]:
                fvs[fvsIndex] = d

            deleteList.append(i)

        # indexがなければ何もしない。  
        except ValueError as e:
            #print(f"Error occurred: {e}")
            pass

    for i in reversed(deleteList):
        # 一度チェックしたら、もうチェックする必要はない。
        del copiedDistances[i]

def minFvsIndex():
    # 仮でfirstNodeをminIndexとしておく。firstNodeのfvはnp.infのはず。
    minIndex = firstNode

    # ここは|V|のオーダ
    for i, fv in enumerate(fvs):
        #print(minIndex)
        if fv[0] < fvs[minIndex][0]:
            minValue = fv[0]
            minIndex = i

    return minIndex

updateFvs(firstNode)
#print("fvs : ", fvs)

# step2 SがVでない場合に処理を行う。
# numSを都度計算しておくことで、時間量を1のオーダにすることができる。
while numS < nodeNum:
    # step3 δ(S)の最小値を求めて、S,Tの更新。
    minIndex = minFvsIndex()
    S[minIndex] = 1
    resultT.append(fvs[minIndex][1])

    # numSが|V|になるまでやるので、このループは|V|のオーダ
    numS += 1
    fvs[minIndex] = [np.inf, []]
    updateFvs(minIndex)

    #print("S : ", S)
    #print("fvs : ", fvs)


print("Tree : ", resultT)


Tree :  [[3, 4], [3, 5], [1, 3], [1, 2]]


In [3]:
# フィボナッチヒープを実装して、プリム法を実装する。

import numpy as np # For np.inf and log2

class FibonacciHeapNode:
    def __init__(self, key, value):
        self.key = key      # The distance/weight to this node from the MST
        self.value = value  # The node index (0-indexed)
        # 直下の子ノードの数を保持するためのフィールド
        self.degree = 0
        self.parent_heap_node = None # Parent in the heap structure
        self.child = None
        # とりあえず自分自身を指すようにしておく（単一ノードの状態）
        self.left = self
        self.right = self
        self.mark = False

class FibonacciHeap:
    def __init__(self):
        self.min_node = None
        self.num_nodes = 0
        self.node_refs = {} # Maps node_value (graph node index) to FibonacciHeapNode object

    # ノードをヒープに挿入する。
    def insert(self, node):
        self.node_refs[node.value] = node
        if self.min_node is None:
            self.min_node = node
        else:
            self._add_to_root_list(node)
            if node.key < self.min_node.key:
                self.min_node = node
        self.num_nodes += 1

    # nodeをルートリストに追加するためのヘルパー関数
    # 最小ノードの右側に追加する。リンクを付け替える
    def _add_to_root_list(self, node):
        # Insert node into the root list (circular doubly linked list)
        if self.min_node:
            node.right = self.min_node.right
            node.left = self.min_node
            self.min_node.right.left = node
            self.min_node.right = node
        else: # This path should primarily be taken when the heap is empty
            # 通常はここは実行されない。
            self.min_node = node
            node.left = node
            node.right = node
        
        # デフォルトでNoneなので、明示的に設定する必要はないが、念のため。
        node.parent_heap_node = None # Root nodes have no heap parent

    # 最小ノードのリンクを切り離して、子ノードをルートリストに移動させる。
    # その後、ルートリストの再構成を行う。
    # ここは|V|のオーダーになる可能性がある。なぜなら、最悪の場合、すべてのノードが同じキーでルートリストに存在する可能性があるから。
    # ただし、平均的にはO(log V)になるはず。
    # 取り出されたノードはヒープから完全に削除されるため、node_refsからも削除する。
    def extract_min(self):
        min_node = self.min_node
        if min_node is not None:
            # Move children of min_node to the root list
            if min_node.child is not None:
                children_to_add = []
                current_child = min_node.child
                # min_nodeの直下の子ノードをすべてルートリストに追加するためのループ
                while True:
                    children_to_add.append(current_child)
                    # ここでcurrent_childを右に移動させて、すべての子ノードを収集する。
                    current_child = current_child.right
                    # もしcurrent_childがmin_nodeのchildに戻ってきたら、すべての子ノードを収集し終わったことになる。
                    if current_child == min_node.child: # Check if we looped back
                        break
                for child in children_to_add:
                    self._add_to_root_list(child)
                    # ルートリストに追加された子ノードは、もはやmin_nodeの子ではないので、親へのリンクを切る。
                    child.parent_heap_node = None # Children become roots
            
            # Remove min_node from root list
            if min_node.right == min_node: # Only one node in root list
                self.min_node = None
            else:
                min_node.left.right = min_node.right
                min_node.right.left = min_node.left
                self.min_node = min_node.right # Set an arbitrary root as potential new min
                self._consolidate()
            self.num_nodes -= 1
            # 取り出されたノードはヒープから完全に削除されるため、node_refsからも削除する。
            # 消されたことでそのノードは探索されたことを示すことになる。
            del self.node_refs[min_node.value]
            return min_node
        return None

    # ルートリストのノードを度数ごとにまとめるための関数
    def _consolidate(self):
        # Determine max degree possible
        # Using log2(n) + 1 for upper bound on degree array size
        max_degree = int(np.log2(self.num_nodes)) + 1 if self.num_nodes > 0 else 0
        # ルートリストのノードを度数ごとにまとめるための配列を用意する
        A = [None] * (max_degree + 1)
        
        # ルートリストのノードを取得する
        root_nodes = []
        if self.min_node is not None:
            current = self.min_node
            while True:
                root_nodes.append(current)
                current = current.right
                if current == self.min_node:
                    break
        
        # ルートリストのノードを度数ごとにまとめる
        # このループは最悪の場合、すべてのノードが同じキーでルートリストに存在する可能性があるため、O(V)になる可能性がある。
        # ただし、平均的にはO(log V)になるはず。
        for w in root_nodes:
            x = w
            d = x.degree

            # 同じ度数のノードが存在する限り、リンクしてまとめる
            # 直下に同じ度数のノードを追加するので度数が1増える。
            while A[d] is not None:
                y = A[d]
                # 度数が同じ2つのノードをつけるが、キーが小さい方を親にする。
                if x.key > y.key: # Ensure x is the one with the smaller key
                    x, y = y, x
                
                self._link(y, x) # Link y to x (y becomes child of x)

                # くっつけて度数が上がるので、元の度数の位置は空にする。
                A[d] = None
                d += 1
            # 空いているところにまとめたノードを置く。
            A[d] = x

        self.min_node = None
        # 最小値を見つけるために、度数ごとにまとめたノードをルートリストに再構築する。
        # ここのループはAのサイズに依存するので、O(log V)になるはず。
        for i in range(len(A)): # Rebuild root list and find new min
            if A[i] is not None:
                # ここは条件分岐をしなくても、_add_to_root_listでmin_nodeがNoneのときに初期化されるようになっている。
                if self.min_node is None: # First non-null node becomes the start of new root list
                    self.min_node = A[i]
                    self.min_node.left = self.min_node
                    self.min_node.right = self.min_node
                else:
                    self._add_to_root_list(A[i]) # Add subsequent nodes to the root list
                    if A[i].key < self.min_node.key:
                        self.min_node = A[i]

    # yをxにリンクする（yがxの子になる）
    def _link(self, y, x): # Link y to x (y becomes a child of x)
        # Remove y from the root list
        # リンクを付け替える
        y.left.right = y.right
        y.right.left = y.left

        # Make y a child of x
        y.parent_heap_node = x
        if x.child is None:
            x.child = y
            y.left = y
            y.right = y
        else:
            y.right = x.child.right
            y.left = x.child
            x.child.right.left = y
            x.child.right = y
        
        # xの子が一つ増えるので＋1する。
        x.degree += 1
        # 新たに子ノードが追加されたyは、まだ一度も子を切り離されていないので、markはFalseのままでよい。
        y.mark = False

    # ノードのキーを減少させる(更新する)関数
    def decrease_key(self, node, new_key):
        if new_key > node.key:
            raise ValueError("new_key is greater than current key")
        
        node.key = new_key
        parent = node.parent_heap_node
        # nodeのキーが親のキーより小さい場合、nodeを親から切り離してルートリストに移動させる。
        if parent is not None and node.key < parent.key:
            self._cut(node, parent)
            self._cascading_cut(parent)
        # nodeは切り取られてルートに移動されているので、nodeのキーが最小ノードのキーより小さい場合は、min_nodeを更新する。
        if node.key < self.min_node.key:
            self.min_node = node

    # nodeを親から切り離してルートリストに移動させる関数
    def _cut(self, x, y): # Cut x from y (x's parent)
        # Remove x from y's child list
        if x.right == x: # x is the only child of y
            y.child = None
        else:
            x.left.right = x.right
            x.right.left = x.left
            if y.child == x: # If x was the designated child, pick another
                y.child = x.right

        # yの子が一つ減るので-1する。
        y.degree -= 1
        
        # Add x to the root list
        self._add_to_root_list(x)
        x.parent_heap_node = None
        x.mark = False

    # マークを使って、親から切り離されたノードが再び切り離される場合に、さらに上の親も切り離すための関数
    # これを利用すると、木の高さがO(log V)に保たれるらしい。
    def _cascading_cut(self, y):
        z = y.parent_heap_node
        if z is not None:
            if not y.mark:
                y.mark = True
            else:
                self._cut(y, z)
                self._cascading_cut(z)

# Prim's Algorithm with Fibonacci Heap
def prim_fibonacci_heap(nodeNum, distances, start_node_idx=0):
    # Convert distances to adjacency list for efficient neighbor lookup
    adj_list = [[] for _ in range(nodeNum)]
    for weight, nodes in distances:
        u, v = nodes[0] - 1, nodes[1] - 1 # Convert to 0-indexed
        adj_list[u].append((v, weight))
        adj_list[v].append((u, weight))

    #min_keys = [np.inf] * nodeNum # Stores the minimum weight to connect node i to the MST
    parent = [None] * nodeNum     # Stores the parent of node i in the MST
    fib_heap = FibonacciHeap()

    # Initialize all nodes with infinity key, except start_node (key 0)
    # このループは|V|のオーダ
    # ここでノードをヒープに挿入する際、start_node_idxのノードだけkeyを0にして、他はnp.infにする。
    for i in range(nodeNum):
        key = 0 if i == start_node_idx else np.inf
        node = FibonacciHeapNode(key, i)
        fib_heap.insert(node)

    mst_edges = []

    while fib_heap.min_node is not None:
        # 最小のノードを抽出する。これがMSTに追加されるノード。
        # ノードはリンクがなくなるが、ノードの値（グラフのノードインデックス）は保持される。
        extracted_node = fib_heap.extract_min()
        # 抽出されたノードの値（グラフのノードインデックス）を取得
        u = extracted_node.value
        #print(f"Extracted node {u + 1} with key {extracted_node.key}") # 1-indexedで表示
        #print("num_nodes : ", fib_heap.num_nodes)

        # If this node has a parent, it means it's connected to the MST via an edge
        if parent[u] is not None:
            # Add the edge (parent[u], u) to the MST. Output in 1-indexed format.
            mst_edges.append([parent[u] + 1, u + 1])

        #print("exist : ", fib_heap.node_refs)

        # Explore neighbors of u
        for v, weight_uv in adj_list[u]:
            # Check if v is still in the heap and if a shorter path is found to v
            # fib_heap.node_refs[v]が存在するかどうかで、vがまだヒープに残っているかを確認する。
            # そのために、extracted_nodeをヒープから完全に削除する際に、node_refsからも削除するようにしている。
            if v in fib_heap.node_refs and weight_uv < fib_heap.node_refs[v].key:
                #print(f"Updating node {v + 1} from key {fib_heap.node_refs[v].key} to {weight_uv}") # 1-indexedで表示
                fib_heap.decrease_key(fib_heap.node_refs[v], weight_uv)
                parent[v] = u # u is now the parent of v in the MST

    return mst_edges

# Use the existing global variables `distances` and `nodeNum`
# The `firstNode` from the previous cell is 3 (representing 4th node, 0-indexed)
firstNode_0_indexed = firstNode 
result_fibonacci_prim = prim_fibonacci_heap(nodeNum, distances, firstNode_0_indexed)

print("Tree (Fibonacci Heap Prim): ", result_fibonacci_prim)

Tree (Fibonacci Heap Prim):  [[4, 3], [3, 5], [3, 1], [1, 2]]


In [4]:
# ライブラリを使った場合も生成した。
import heapq
import numpy as np

def prim_heapq(nodeNum, distances, start_node_idx=0):
    # グラフの隣接リストを作成
    adj_list = [[] for _ in range(nodeNum)]
    # ここのループは|E|のオーダー
    for weight, nodes in distances:
        u, v = nodes[0] - 1, nodes[1] - 1 # 0-indexed に変換
        adj_list[u].append((v, weight))
        adj_list[v].append((u, weight))

    min_cost = [np.inf] * nodeNum  # MSTへの各ノードの最小コストを保存
    parent = [None] * nodeNum     # MST内での各ノードの親を保存
    visited = [False] * nodeNum   # ノードがMSTに含まれているかどうかのフラグ

    # 優先度キュー (コスト, ノードインデックス) を格納
    # heapqは最小ヒープなので、コストが最小の要素が常に取り出される
    priority_queue = [(0, start_node_idx)] # (コスト, ノード)
    min_cost[start_node_idx] = 0

    mst_edges = []

    # 優先度キューが空になるまでループ
    while priority_queue:
        # コストが最小のノードを取り出す
        cost, u = heapq.heappop(priority_queue)

        # すでに訪問済みのノードはスキップ
        if visited[u]:
            continue

        # ノードuをMSTに追加
        visited[u] = True

        # MSTに追加されるエッジを記録 (開始ノードでない場合)
        if parent[u] is not None:
            # 1-indexed に戻して保存
            mst_edges.append([parent[u] + 1, u + 1])

        # 隣接ノードを探索
        # ノードuの隣接ノードで距離が小さいものを優先度キューに追加
        for v, weight_uv in adj_list[u]:
            # vがまだMSTに含まれておらず、uを経由するコストが現在の最小コストより小さい場合
            if not visited[v] and weight_uv < min_cost[v]:
                min_cost[v] = weight_uv
                parent[v] = u
                # ここの計算量は理想的にはO(log V)ですが、heapqは減少キー操作をサポートしていないため、単純に新しいエントリを追加する形になります。
                heapq.heappush(priority_queue, (weight_uv, v))

    return mst_edges

# 既存のグローバル変数 `distances` と `nodeNum` を使用
# `firstNode` は1-indexedなので、0-indexedに変換
start_node_0_indexed = firstNode # previous cell defines firstNode as 3 (0-indexed 3 is 4th node)
result_heapq_prim = prim_heapq(nodeNum, distances, start_node_0_indexed)

print("Tree (Heapq Prim): ", result_heapq_prim)


Tree (Heapq Prim):  [[4, 3], [3, 5], [3, 1], [1, 2]]
